# U-Net 3D V100 Analysis 

## Notebook Overview

This notebook presents an analysis of a U-Net 3D workload using the DFAnalyzer tool. It demonstrates how to analyze I/O traces collected from a deep learning application running on a V100 system. The workflow includes:

- Setting up the environment and importing necessary libraries.
- Extracting and preparing trace data for analysis.
- Initializing DFAnalyzer with appropriate configuration.
- Running the analysis to generate summarized I/O statistics and views.
- Displaying and interpreting the results, including bandwidth and operation counts over time ranges for different I/O layers.

The notebook is intended to help users understand the I/O behavior of deep learning workloads and provides a template for similar analyses on other datasets.

## Interactive Analysis

### Prepare Environment

In this section, we set up the environment by importing required libraries, configuring warning filters, and updating the Python path to include the DFAnalyzer workspace. 

In [ ]:
import os
import sys
import warnings

# Add DFAnalyzer to the path
workspace_dir = os.path.abspath("/g/g92/marathe1/myworkspace/dldl/dfanalyzer/")
sys.path.append(workspace_dir)

# Filter warnings
warnings.filterwarnings('ignore')

### Prepare Trace Data

Then, we extract the trace data archive into the designated directory to prepare it for analysis with DFAnalyzer.

In [ ]:
!mkdir -p {workspace_dir}/tests/data/extracted/dftracer-dlio
!tar -xzf {workspace_dir}/tests/data/dftracer-dlio.tar.gz -C {workspace_dir}/tests/data/extracted/dftracer-dlio

In [2]:
import sys, shutil, os, pathlib

# Install the build tools into THIS interpreter
!{sys.executable} -m pip install -q --upgrade pip setuptools wheel
!{sys.executable} -m pip install -q cmake ninja scikit-build

# Ensure the kernel's bin dir is on PATH (fixes earlier PATH issues)
bin_dir = str(pathlib.Path(sys.executable).parent)
if shutil.which("cmake") is None:
    os.environ["PATH"] = bin_dir + ":" + os.environ.get("PATH","")

print("python:", sys.executable)
print("PATH[0:2]:", os.environ["PATH"].split(":")[:2])
!which cmake
!cmake --version
!which ninja

python: /g/g92/marathe1/my_personal_env/bin/python3
PATH[0:2]: ['/g/g92/marathe1/my_personal_env/bin', '/usr/apps/python-3.12.2/bin/python3']
/bin/bash: which: command not found
cmake version 4.1.0

CMake suite maintained and supported by Kitware (kitware.com/cmake).
/bin/bash: which: command not found


In [1]:
import sys, shlex
cmd = f"""
source /usr/share/lmod/lmod/init/bash || source /usr/share/Modules/init/bash
module load gcc/12.1.1
module load cmake
module load binutils   # for ar/nm/ranlib if needed
which gcc; which cmake; which ar || true
{shlex.quote(sys.executable)} -m pip install --no-build-isolation --no-cache-dir zindex_py
"""
get_ipython().run_line_magic("bash", f"-lc {shlex.quote(cmd)}")

UsageError: Line magic function `%bash` not found (But cell magic `%%bash` exists, did you mean that instead?).


### Run Analysis

Finaly, we initialize the DFAnalyzer with the specified configuration and run the trace analysis to generate summarized I/O statistics and views for further exploration.

In [ ]:
from dfanalyzer import init_with_hydra

percentile = 0.9
run_dir = f"{workspace_dir}/notebooks/.dfanalyzer/unet3d_v100_hdf5"
time_granularity = 5  # 5 seconds
trace_path = f"{workspace_dir}/tests/data/extracted/dftracer-dlio"
view_types = ["time_range", "proc_name"]

dfa = init_with_hydra(
    hydra_overrides=[
        'analyzer=dftracer',
        'analyzer/preset=dlio',
        'analyzer.checkpoint=False',
        f"analyzer.time_granularity={time_granularity}",
        f"hydra.run.dir={run_dir}",
        f"percentile={percentile}",
        f"trace_path={trace_path}",
    ]
)

ModuleNotFoundError: No module named 'dfanalyzer'

We access the underlying Dask client via our Python API.

In [ ]:
dfa.client

We access to current preset configuration as follows.

In [ ]:
dict(dfa.analyzer.preset.layer_defs)

We run the analysis via the `analyze_trace` function as follows.

In [ ]:
result = dfa.analyze_trace(percentile=percentile, view_types=view_types)

And, using the `output` variable available in our analyzer instance `dfa`, we output the DFAnalyzer summary.

In [ ]:
dfa.output.handle_result(result)

### Result Exploration

We access the high-level characteristics and layer-based characteristics and metrics via our Python API as follows:

In [ ]:
result._traces.head()

In [ ]:
result._hlms.keys()

In [ ]:
result._main_views['reader_posix_lustre'].head()

In [ ]:
result.views['reader_posix_lustre'][('time_range',)].head()